In [2]:
import pandas as pd
import numpy as np
import os

input_file = "data/weather/YSSY-Winds/Data Processing/ProcessedData/YSSY.txt"

output_folder = "data/weather/YSSY-Winds/Data Processing/InterpolatedData0.5"
output_file = os.path.join(output_folder, "YSSY.txt")

os.makedirs(output_folder, exist_ok=True)

print("Input file:", input_file)
print("Output file:", output_file)

Input file: data/weather/YSSY-Winds/Data Processing/ProcessedData/YSSY.txt
Output file: data/weather/YSSY-Winds/Data Processing/InterpolatedData0.5/YSSY.txt


In [3]:
df = pd.read_csv(
    input_file,
    parse_dates=["timestamp"],
    na_values=["NaN"]
)

print(df.shape)
df.head()

(796079, 7)


,timestamp,air_temp,dew_point,wind_speed,wind_dir,msl_pressure,data_completeness
0,1980-01-02 00:30:00,19.0,16.0,1.9,120.0,1017.0,1
1,1980-01-02 01:00:00,18.0,17.0,1.9,110.0,1017.0,1
2,1980-01-02 01:30:00,19.0,17.0,2.9,110.0,1017.0,1
3,1980-01-02 02:00:00,18.0,17.0,2.9,110.0,1017.0,1
4,1980-01-02 02:30:00,18.0,16.0,2.9,120.0,1017.0,1


## Load YSSY Processed Weather Data

In this step, the processed weather data file for station **YSSY** is loaded from the `ProcessedData` folder.

The file is read using `pd.read_csv()`, and the `timestamp` column is parsed as a datetime object. Missing values written as `"NaN"` in the file are also recognised as real missing values by pandas.

The loaded dataset contains **796,079 rows and 7 columns**.

The columns are:

- `timestamp`: observation time
- `air_temp`: air temperature
- `dew_point`: dew point temperature
- `wind_speed`: wind speed
- `wind_dir`: wind direction
- `msl_pressure`: mean sea level pressure
- `data_completeness`: completeness flag for each row

The first few records show that the dataset starts from **1980-01-02 00:30:00**, with observations recorded at regular 30-minute intervals.

In [4]:
df.isna().sum()

timestamp                0
air_temp             21581
dew_point            21578
wind_speed           20594
wind_dir             20510
msl_pressure         24103
data_completeness        0
dtype: int64

In [6]:
ELEMENTS_TO_INTERPOLATE = [

    "air_temp",

    "dew_point",

    "wind_speed",

    "wind_dir",

    "msl_pressure"

]

print(ELEMENTS_TO_INTERPOLATE)

['air_temp', 'dew_point', 'wind_speed', 'wind_dir', 'msl_pressure']


In [7]:
def interpolate_single_step_gaps(series):

    interpolated_series = series.copy()

    is_na = series.isna()

    n = len(series)

    gaps_interpolated_count = 0

    for i in range(1, n - 1):

        if not is_na.iloc[i - 1] and is_na.iloc[i] and not is_na.iloc[i + 1]:

            interpolated_value = (series.iloc[i - 1] + series.iloc[i + 1]) / 2.0

            interpolated_series.iloc[i] = interpolated_value

            gaps_interpolated_count += 1

    print(f"{series.name}: interpolated {gaps_interpolated_count} single-step gaps.")

    return interpolated_series

df_interpolated = df.copy()

for element in ELEMENTS_TO_INTERPOLATE:

    if element in df_interpolated.columns:

        df_interpolated[element] = pd.to_numeric(df_interpolated[element], errors="coerce")

        df_interpolated[element] = interpolate_single_step_gaps(df_interpolated[element])

    else:

        print(f"Warning: {element} not found, skipped.")

df_interpolated.head()

air_temp: interpolated 15451 single-step gaps.
dew_point: interpolated 15466 single-step gaps.
wind_speed: interpolated 14970 single-step gaps.
wind_dir: interpolated 14984 single-step gaps.
msl_pressure: interpolated 16274 single-step gaps.


,timestamp,air_temp,dew_point,wind_speed,wind_dir,msl_pressure,data_completeness
0,1980-01-02 00:30:00,19.0,16.0,1.9,120.0,1017.0,1
1,1980-01-02 01:00:00,18.0,17.0,1.9,110.0,1017.0,1
2,1980-01-02 01:30:00,19.0,17.0,2.9,110.0,1017.0,1
3,1980-01-02 02:00:00,18.0,17.0,2.9,110.0,1017.0,1
4,1980-01-02 02:30:00,18.0,16.0,2.9,120.0,1017.0,1


## Apply Single-Step Gap Interpolation

In this step, the single-step gap interpolation function was applied to each selected weather variable.

The function checks each value in the time series and only fills missing values where the missing value is located between two valid neighbouring values. For example:

`valid value, NaN, valid value`

The missing value is replaced by the average of the previous and next values.

The interpolation results are:

| Variable | Number of Single-Step Gaps Filled |
|---|---:|
| air_temp | 15,451 |
| dew_point | 15,466 |
| wind_speed | 14,970 |
| wind_dir | 14,984 |
| msl_pressure | 16,274 |

This shows that the interpolation successfully filled a large number of isolated missing values in the YSSY dataset.

After interpolation, the first five rows of the dataset remain complete, with `data_completeness = 1` for each row shown. This is expected because these rows did not contain missing values before interpolation.

In [8]:
cols_for_completeness_check = [

    col for col in ELEMENTS_TO_INTERPOLATE

    if col in df_interpolated.columns

]

df_interpolated["data_completeness"] = (

    df_interpolated[cols_for_completeness_check].notna().all(axis=1)

    & df_interpolated["timestamp"].notna()

).astype(int)

print("Columns used for completeness check:")

print(cols_for_completeness_check)

print("\nFirst 5 rows after recalculating data_completeness:")

display(df_interpolated.head())

Columns used for completeness check:
['air_temp', 'dew_point', 'wind_speed', 'wind_dir', 'msl_pressure']

First 5 rows after recalculating data_completeness:


,timestamp,air_temp,dew_point,wind_speed,wind_dir,msl_pressure,data_completeness
0,1980-01-02 00:30:00,19.0,16.0,1.9,120.0,1017.0,1
1,1980-01-02 01:00:00,18.0,17.0,1.9,110.0,1017.0,1
2,1980-01-02 01:30:00,19.0,17.0,2.9,110.0,1017.0,1
3,1980-01-02 02:00:00,18.0,17.0,2.9,110.0,1017.0,1
4,1980-01-02 02:30:00,18.0,16.0,2.9,120.0,1017.0,1


In [9]:
comparison = pd.DataFrame({
    "before": df[ELEMENTS_TO_INTERPOLATE].isna().sum(),
    "after": df_interpolated[ELEMENTS_TO_INTERPOLATE].isna().sum()
})

comparison["filled"] = comparison["before"] - comparison["after"]

print("Missing value comparison before and after interpolation:")
display(comparison)

Missing value comparison before and after interpolation:


,before,after,filled
air_temp,21581,6130,15451
dew_point,21578,6112,15466
wind_speed,20594,5624,14970
wind_dir,20510,5526,14984
msl_pressure,24103,7829,16274


## Missing Value Comparison Result

The comparison table shows the number of missing values before and after applying single-step gap interpolation.

| Variable | Missing Before | Missing After | Filled |
|---|---:|---:|---:|
| air_temp | 21,581 | 6,130 | 15,451 |
| dew_point | 21,578 | 6,112 | 15,466 |
| wind_speed | 20,594 | 5,624 | 14,970 |
| wind_dir | 20,510 | 5,526 | 14,984 |
| msl_pressure | 24,103 | 7,829 | 16,274 |

The `filled` column shows how many missing values were successfully filled by interpolation.

For example, `air_temp` originally had **21,581** missing values. After interpolation, this was reduced to **6,130**, meaning **15,451** single-step gaps were filled.

Overall, the interpolation step significantly reduced missing values across all selected weather variables. However, some missing values remain because the method only fills isolated one-step gaps. Consecutive missing values or missing values at the beginning or end of the dataset are not interpolated.

## Reflection on the Interpolation Code

This code was used to test single-step gap interpolation on the processed YSSY weather dataset. Overall, the code works correctly for its intended purpose. It successfully loads the processed data, applies interpolation to selected numerical weather variables, recalculates the `data_completeness` flag, compares missing values before and after interpolation, and saves the cleaned output file.

The main strength of this code is that it uses a conservative interpolation method. It only fills isolated one-step missing values, where a `NaN` is surrounded by two valid values. This is useful for weather data because it avoids making strong assumptions over long missing periods. Longer gaps may represent sensor problems, station outages, or unreliable data periods, so leaving them unchanged is safer than filling them automatically.

The result also shows that the method is effective for this dataset. A large number of missing values were filled across all selected variables, especially for `air_temp`, `dew_point`, `wind_speed`, `wind_dir`, and `msl_pressure`. At the same time, some missing values remained, which is expected because the function does not fill consecutive gaps or gaps at the beginning or end of the dataset.

However, there are several possible improvements to consider.

First, the interpolation method assumes that taking the average of the previous and next values is appropriate for all variables. This is reasonable for variables like air temperature, dew point, wind speed, and pressure, but wind direction is more complicated. Wind direction is circular data, meaning that 0 degrees and 360 degrees represent almost the same direction. For example, interpolating between 350 degrees and 10 degrees using a normal average gives 180 degrees, which is incorrect. A circular interpolation method would be more appropriate for `wind_dir`.

Second, the code currently uses a manual loop through each row. This is easy to understand, but it may be slower for very large datasets. Since the YSSY file has 796,079 rows, the runtime is still acceptable, but if many stations are processed together, a more vectorised pandas approach could improve efficiency.

Third, the code only checks whether values are missing, but it does not check whether the values are physically reasonable. For example, extreme wind speeds, negative pressure values, or invalid wind directions could still remain in the dataset if they are not marked as `NaN`. A future improvement could include basic range checks before interpolation.

Fourth, the `data_completeness` column is recalculated after interpolation, which is useful. However, it may also be helpful to keep another flag showing whether a value was originally observed or interpolated. For example, an extra column such as `air_temp_interpolated_flag` could help later modelling or analysis distinguish between real observations and filled values.

Fifth, the code currently applies the same interpolation rule to all selected variables. In future work, different weather variables could have different cleaning rules. For example, pressure and temperature may be suitable for simple interpolation, while wind direction may need circular interpolation, and wind speed may need extra checks for calm wind conditions.

Overall, the current code is a good first-pass data cleaning step. It is simple, transparent, and avoids overfilling large missing periods. The main improvement would be to handle wind direction more carefully and to add quality-control flags so that later analysis can track which values were original and which values were interpolated.